### SkyScout: Fine-Tuning and Benchmarking YOLO and Transformer Detectors for Small-Object Detection in Drone Imagery

Francis Neil Moral, TensorFlow II project, Fanshawe College

This notebook runs the full project. The code is in `skyscout.py` and every setting is in `config.yaml`.
Each cell can be run again after a disconnect: finished steps are skipped and training continues from the last saved epoch.

#### 1. Setup

Finds the machine type (Colab, server or local PC) by itself.

In [ ]:
import os
import subprocess
import sys

# Recommended: on Colab, upload the whole project folder to Google Drive as MyDrive/SkyScout
PROJECT_DIR = '/content/drive/MyDrive/SkyScout'

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir(PROJECT_DIR)
    # Recommended: if this upgrades TensorFlow, restart the session once and run this cell again
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
sys.path.insert(0, os.getcwd())

from IPython.display import Image, Markdown, display
import skyscout as sk

print('Profile:', sk.PROFILE, '| GPU:', sk.gpu_name())
sk.show_status()

#### 2. Dataset

VisDrone2019-DET is downloaded once into `downloads/`. Ignored regions and the "others" class are dropped.

In [ ]:
sk.prepare_dataset()
sk.dataset_summary()

#### 3. Time one epoch

Estimate the training time before starting a full run. Results go to `results/timing.json`.

In [ ]:
sk.time_epoch('yolo11s_640')
sk.time_epoch('dfine_full_640')

#### 4. Experiment 1: YOLO11s baseline at 640

In [ ]:
sk.run_experiment('yolo11s_640')
display(Markdown(sk.eval_dir('yolo11s_640', 'test').joinpath('report.md').read_text()))

#### 5. Experiment 2: YOLO26s at 640

In [ ]:
sk.run_experiment('yolo26s_640')
display(Markdown(sk.eval_dir('yolo26s_640', 'test').joinpath('report.md').read_text()))

#### 6. Experiment 3: D-FINE fine-tuning setups

Three setups, 30 epochs each: frozen backbone, full fine-tune, and full fine-tune with layer-wise learning rate decay.
The best one (by val mAP50-95) is then trained longer.

In [ ]:
for name in sk.CFG['dfine_setup_compare']:
    sk.run_experiment(name)
print('Best setup:', sk.best_dfine())

In [ ]:
sk.run_experiment('dfine_best_long_640')
display(Markdown(sk.eval_dir('dfine_best_long_640', 'test').joinpath('report.md').read_text()))

#### 7. Experiment 4: input size 640 vs 1024

The 1024 runs are set to the server profile in `config.yaml`. On other machines these cells only print a recommendation.

In [ ]:
for name in ['yolo11s_1024', 'yolo26s_1024', 'dfine_best_1024']:
    sk.run_experiment(name)

#### 8. Experiment 5: SAHI tiled inference

In [ ]:
sk.run_sahi()

#### 9. Experiment 6: data efficiency (D-FINE on 25%, 50%, 100% of the training set)

In [ ]:
for name in ['dfine_best_25', 'dfine_best_50']:
    sk.run_experiment(name)

#### 10. Results

In [ ]:
sk.make_report()
display(Markdown(sk.RESULTS.joinpath('comparison.md').read_text().split('![')[0]))
for p in sorted(sk.PLOTS.glob('*.png')):
    display(Markdown(f'**{p.stem}**'), Image(str(p)))

#### 11. Demo

Upload a drone image and get boxes plus a count per class. On Colab a public link is printed.

In [ ]:
sk.launch_demo()